# Exp 3 debug · Test E only, 5 clips (T11)

Standalone version of `exp3_debug.ipynb` Cell 9: **DINOv2-B features, bilinear and AnyUp2D upsampling, label propagation at three resolutions, plus the published DINO protocol as an external reference.** It needs no other notebook.

| Variant | DINOv2 input | Propagation grid |
|---|---|---|
| `e224_p56` | 224² (squashed, as Exp 3) | 56² |
| `e448_p112` | 448² (squashed) | 112² |
| `e448x798_p112x200` | 448×798 (aspect kept) | 112×200 |
| `dino_ref` | 476×854 (aspect kept) | native 34×61 tokens, no upsampler (DINO protocol) |

**Run:** T4 runtime, then Run all. **Runtime → Restart session** first if `anyup` was already imported. Cell 4 runs the tests and Cell 5 draws the visuals. If you interrupt Cell 4 (Runtime → Interrupt, **not** restart), Cell 5 still shows every clip that finished. Rough time with the 5 default clips on a T4 (estimate from the 30-clip run): ≈ 35–45 min, mostly `e448x798_p112x200`. Remove that variant in Cell 2 for ≈ 10–15 min.

In [ ]:
# ── Cell 1 · Repo + Drive + deps ────────────────────────────────────────────
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # deterministic cuBLAS (T07); must be set before anything starts CUDA
import sys, subprocess
from google.colab import drive
drive.mount("/content/drive")

REPO_URL    = "https://github.com/mu-az88/anyup.git"
REPO_BRANCH = "experiments-in-action-test"   # T11
REPO_DIR    = "/content/anyup"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    # existing clone may be on another branch — switch to REPO_BRANCH and update it
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True)
if "anyup" in sys.modules:
    raise RuntimeError("anyup was already imported in this runtime — Runtime → Restart session, then run again")
subprocess.run(["pip", "install", "-e", REPO_DIR, "-q"], check=True)
subprocess.run(["pip", "install", "transformers", "timm", "scikit-image", "scipy", "-q"], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# ── Cell 2 · Config ─────────────────────────────────────────────────────────
CLIPS_E      = ["blackswan", "dog", "bmx-trees", "drift-chicane", "parkour"]
                       # ↓ clips (time ∝ their total frames, printed in Cell 3). Chosen to cover easy clips (blackswan, dog), a thin
                       #   object (bmx-trees), the clip that vanished everywhere in run 1 (drift-chicane) and the AnyUp3D collapse (parkour)
OUTPUT_ROOT  = "/content/drive/MyDrive/results/T11/exp3_debug_E"
DAVIS_DEST   = "/content"
DEVICE       = "cuda"
SEED         = 0

E_CONDITIONS = ["bilinear", "anyup2d"]   # simple upsamplers only (AnyUp3D's full attention does not fit these sizes on a T4)
E_VARIANTS   = [   # (name, DINOv2 input (h, w), propagation grid (h, w)); each = one DINOv2 pass + one propagation per condition and clip
    ("e224_p56",          (224, 224), (56, 56)),     # = Exp 3 sizes with DINOv2 instead of Swin-T
    ("e448_p112",         (448, 448), (112, 112)),   # ↓ 2× finer grid, still squashed; propagation ≈ 16× the 56² cost
    ("e448x798_p112x200", (448, 798), (112, 200)),   # ↓ aspect kept (480×854 × 0.93; 798 = 57·14). Clip features on GPU = T·22400·768·4 B
]                                                    #   (≈ 69 MB per frame: ≈ 6.9 GB for a 100-frame clip). Dependent: E_Q_CHUNK, E_CHUNK, CLIPS_E lengths
TAU, TOP_K   = 0.07, 7 # Exp 3 propagation temperature / neighbours (E_VARIANTS)
N_LAST       = 6       # ↓ context frames besides frame 0 (Exp 3). Dependent: E_Q_CHUNK (block width = (1+N_LAST)·h·w)
E_CHUNK      = 8       # ↓ frames per DINOv2 / AnyUp2D call (DINOv2 attention ∝ tokens², 1824 tokens at 448×798). Dependent: E_VARIANTS inputs
E_Q_CHUNK    = 1024    # ↓ query pixels per propagation block; block = E_Q_CHUNK × (1+N_LAST)·h·w floats (112×200: ≈ 640 MB).
                       #   Dependent: E_VARIANTS grids, N_LAST, E_REF n_last
RUN_REF      = True    # DINO reference protocol (small 34×61 grid: cheap)
E_REF        = dict(input=(476, 854), tau=0.1, top_k=5, n_last=7, radius=12)
                       # DINO protocol (Caron et al. 2021, eval_video_segmentation.py): 480p aspect kept (476×854 = multiples of 14)
                       # → native 34×61 token grid, no upsampler; window 12 cells, top-5, τ 0.1, frame 0 + last 7; soft labels ×14 bilinear → argmax
E_REF_PAPER  = 0.623   # DINO ViT-B/16 DAVIS-2017 val J&F (paper, 30 clips; different model: order of magnitude only)

SHOW_FRACS   = [0.5, 0.9]   # mask figure: frames at these fractions of each clip (errors grow over time); display only
SHOW_H       = 160     # panel height in px (width keeps the aspect ratio); display only

In [ ]:
# ── Cell 3 · Imports, data, helpers ─────────────────────────────────────────
import time, json, gc
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from anyup import davis, metrics, viz
from anyup.backbones import BackboneLoader
from anyup.video import iter_backbone_chunks
from anyup.upsamplers import UpsamplerLoader
from anyup.utils.seed import set_seed

set_seed(SEED)
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)
ROOT = davis.download_davis(DAVIS_DEST)
missing = [c for c in CLIPS_E if c not in davis.list_clips(ROOT, "val")]
assert not missing, f"not DAVIS-2017 val clips: {missing}"
CLIPS = list(CLIPS_E)
OUT_DIR = Path(OUTPUT_ROOT) / f"n{len(CLIPS)}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"{len(CLIPS)} clips | frames: " + ", ".join(f"{c} {len(davis.frame_paths(ROOT, c))}" for c in CLIPS) + f" | outputs → {OUT_DIR}")

E_UPS = {c: UpsamplerLoader.load(c, device=DEVICE) for c in E_CONDITIONS}
RES_E, PREDS_E, GRIDS_E, SKIPPED_E = {}, {}, {}, {}   # (variant, cond) → clip → eval | (T, h, w) uint8 preds; variant → grid; variant → reason


def clip_mean_jf(res):
    return (float(np.mean(res["J_M"])) + float(np.mean(res["F_M"]))) / 2


def propagate_e(feats_flat, hw, ann0, tau, top_k, n_last, radius=None, q_chunk=E_Q_CHUNK):
    """
    The davis.propagate_labels rule (Exp 3), computed in query chunks (memory ∝ q_chunk, not h·w): each query pixel keeps
    its top_k context matches (optionally only within `radius` grid cells, DINO's size_mask_neighborhood), softmax over
    those top_k with temperature tau, label = weighted mean of their soft labels.
    feats_flat (T, N, C) unit vectors on GPU; hw = (h, w), h·w = N; ann0 (H0, W0) GT of frame 0. → soft labels (T, n_ch, h, w).
    """
    T, N, C = feats_flat.shape
    h, w = hw
    n_objects = int(ann0.max())
    n_ch = n_objects + 1
    labels = torch.zeros(T, n_ch, N, device=feats_flat.device)                          # (T, n_ch, N)
    labels[0] = davis._label_maps_frame0(ann0, n_objects, h, w).to(feats_flat.device)
    if radius is not None:
        yy, xx = torch.meshgrid(torch.arange(h, device=feats_flat.device),
                                torch.arange(w, device=feats_flat.device), indexing="ij")
        pos = torch.stack([yy.flatten(), xx.flatten()], 1)                               # (N, 2) grid position
    for t in range(1, T):
        ctx = [0] + list(range(max(1, t - n_last), t))                                   # frame 0 always first
        k_feats = feats_flat[ctx].reshape(-1, C)                                         # (Nctx·N, C), frame-major
        k_labels = labels[ctx].permute(1, 0, 2).reshape(n_ch, -1)                        # (n_ch, Nctx·N)
        for q0 in range(0, N, q_chunk):
            q1 = min(N, q0 + q_chunk)
            A = feats_flat[t, q0:q1] @ k_feats.T / tau                                   # (Q, Nctx·N)
            if radius is not None:
                near = (pos[q0:q1, None, :] - pos[None, :, :]).abs().amax(-1) <= radius  # (Q, N) square window
                A = A.masked_fill(~near.repeat(1, len(ctx)), float("-inf"))
            vals, idx = A.topk(top_k, dim=-1)                                            # (Q, top_k)
            wts = F.softmax(vals, dim=-1)                                                # (Q, top_k)
            labels[t, :, q0:q1] = (k_labels[:, idx] * wts[None]).sum(-1)                 # (n_ch, Q, top_k) → (n_ch, Q)
            del A
    return labels.view(T, n_ch, h, w)


def clip_feats(bb, clip, cond, out_hw):
    """DINOv2 on every frame (E_CHUNK frames per call) → upsampled to out_hw by `cond` (None = native token grid)
    → L2-normalized, written chunk by chunk into one (T, N, C) GPU tensor (no second full-size copy)."""
    flat, t0 = None, 0
    for ch in iter_backbone_chunks(bb, davis.open_clip(ROOT, clip), chunk_size=E_CHUNK, overlap=0):
        f = ch.features if cond is None else E_UPS[cond](ch, output_size=out_hw).features   # (t, C, h, w)
        t, C, h, w = f.shape
        if flat is None:
            flat = torch.empty(len(davis.frame_paths(ROOT, clip)), h * w, C, device=f.device)   # (T, N, C)
        flat[t0:t0 + t] = F.normalize(f.reshape(t, C, h * w).permute(0, 2, 1), dim=-1)  # (t, N, C) unit vectors
        t0 += t
        del f
    assert t0 == flat.shape[0], "frame mismatch"
    return flat, (h, w)


def to_native(pred_lr, gt):
    return np.stack([davis.resize_labels(np.asarray(p, np.uint8), gt.shape[1:]) for p in pred_lr])   # (T, H0, W0), nearest


def run_e(name, input_hw, prop_hw, conds, ref=False):
    """One variant over all clips: per clip and condition DINOv2 → (upsampler) → propagation → official J&F."""
    bb = BackboneLoader.load("dinov2", device=DEVICE, input_size=input_hw,
                             guide_size=prop_hw if prop_hw else input_hw, variant="base")
    for c in conds:
        RES_E.setdefault((name, c), {}); PREDS_E.setdefault((name, c), {})
    t_start = time.perf_counter()
    try:
        for ci, clip in enumerate(CLIPS):
            t0 = time.perf_counter()
            gt = davis.load_annotations(ROOT, clip)                                     # (T, 480, W) uint8
            line = []
            for c in conds:   # backbone re-run per condition: holding two full-size feature sets would double GPU memory
                flat, hw = clip_feats(bb, clip, None if ref else c, prop_hw)
                GRIDS_E[name] = hw
                if ref:
                    soft = propagate_e(flat, hw, gt[0], E_REF["tau"], E_REF["top_k"], E_REF["n_last"], E_REF["radius"])
                    pred = torch.stack([F.interpolate(s[None], scale_factor=14, mode="bilinear", align_corners=False)[0].argmax(0)
                                        for s in soft]).cpu().numpy()                   # (T, 14h, 14w), as DINO
                else:
                    soft = propagate_e(flat, hw, gt[0], TAU, TOP_K, N_LAST)
                    pred = soft.argmax(1).cpu().numpy()                                 # (T, h, w), as Exp 3
                    if ci == 0 and c == conds[0] and name == E_VARIANTS[0][0]:          # check: same rule as davis
                        T_, N_, C_ = flat.shape
                        ref_pred = davis.propagate_labels(flat.permute(0, 2, 1).reshape(T_, C_, *hw), gt[0],
                                                          tau=TAU, top_k=TOP_K, n_last=N_LAST).numpy()
                        nd = int((ref_pred != pred).sum())
                        print(f"check propagate_e vs davis.propagate_labels on {clip}: {nd} of {pred.size} "
                              f"grid pixels differ → {'PASS' if nd <= 1e-4 * pred.size else 'WARN'}")
                del flat, soft
                torch.cuda.empty_cache()
                RES_E[(name, c)][clip] = metrics.eval_clip_jf(gt, to_native(pred, gt))
                PREDS_E[(name, c)][clip] = pred.astype(np.uint8)
                line.append(f"{c} {clip_mean_jf(RES_E[(name, c)][clip]):.3f}")
            print(f"[{ci + 1}/{len(CLIPS)}] {clip:<16} {name}: " + "  ".join(line) + f"  {time.perf_counter() - t0:.0f}s")
    except torch.cuda.OutOfMemoryError as e:
        SKIPPED_E[name] = f"CUDA out of memory on {GPU_NAME} ({str(e).splitlines()[0][:120]}); reduce E_CHUNK / E_Q_CHUNK"
        print(f"SKIPPED rest of {name}: {SKIPPED_E[name]}")
    del bb
    gc.collect(); torch.cuda.empty_cache()
    print(f"{name}: grid {GRIDS_E.get(name)}, {time.perf_counter() - t_start:.0f}s\n")


def frame_and_gt(clip, t):
    """Native frame t + its GT, resized to height SHOW_H (aspect kept)."""
    img = np.array(Image.open(davis.frame_paths(ROOT, clip)[t]).convert("RGB"))
    hw = (SHOW_H, int(round(img.shape[1] * SHOW_H / img.shape[0])))
    gt = davis.load_annotation(davis.ann_paths(ROOT, clip)[t])
    return viz.resize_rgb(img, hw), davis.resize_labels(gt, hw), hw

In [ ]:
# ── Cell 4 · Run Test E ──────────────────────────────────────────────────────
# Interrupting this cell (Runtime → Interrupt, NOT restart) keeps every finished clip; Cell 5 then shows them.
t_all = time.perf_counter()
if RUN_REF:   # first: cheap, and it is the external check of the propagation + scoring code
    run_e("dino_ref", E_REF["input"], None, ["none"], ref=True)
for name, input_hw, prop_hw in E_VARIANTS:
    run_e(name, input_hw, prop_hw, E_CONDITIONS)
print(f"Test E done in {time.perf_counter() - t_all:.0f}s")

In [ ]:
# ── Cell 5 · Results + visuals (works on whatever Cell 4 finished) ──────────
import sys
sys.last_traceback = sys.last_value = sys.last_type = None   # after an interrupt the traceback still holds GPU tensors
gc.collect(); torch.cuda.empty_cache()

order = (["dino_ref"] if RUN_REF else []) + [n for n, _, _ in E_VARIANTS]
keys = [k for n in order for k in RES_E if k[0] == n and RES_E[k]]
clips_of = {k: [c for c in CLIPS if c in RES_E[k]] for k in keys}
common = [c for c in CLIPS if all(c in RES_E[k] for k in keys)]   # clips finished by every variant → fair comparison
colors = {"bilinear": "0.55", "anyup2d": "tab:orange", "none": "tab:purple"}
label = lambda k: "DINO protocol" if k[1] == "none" else f"{k[0]} · {k[1]}"


def lost(k, clips):
    """(frame, object) pairs scored by the toolkit where the object is in the GT but has 0 predicted pixels."""
    l = p = 0
    for clip in clips:
        gt = davis.load_annotations(ROOT, clip)
        nat = to_native(PREDS_E[k][clip], gt)
        for t in range(1, len(gt) - 1):
            for o in range(1, int(gt[0][gt[0] != 255].max()) + 1):
                if (gt[t] == o).any():
                    p += 1
                    l += int(not (nat[t] == o).any())
    return l, p


CEIL = {}
def ceiling(hw, clips):
    """Test A rule: GT → grid → native (nearest) → J&F = best score a perfect propagation could get on that grid."""
    key = (tuple(hw), tuple(clips))
    if key not in CEIL:
        ce = {}
        for clip in clips:
            gt = davis.load_annotations(ROOT, clip)
            rt = np.stack([davis.resize_labels(davis.resize_labels(m, hw), gt.shape[1:]) for m in gt])
            rt[rt == 255] = 0
            ce[clip] = metrics.eval_clip_jf(gt, rt)
        CEIL[key] = metrics.summarize_jf(ce)["J&F-Mean"]
    return CEIL[key]


# ── table (official J&F over the objects of the clips finished by every variant) ──
SUM = {k: metrics.summarize_jf({c: RES_E[k][c] for c in common}) for k in keys}
print(f"Official J&F on {len(common)} clips {common}")
print(f"{'variant':<20} {'cond':<9} {'grid':>10} {'J&F':>7} {'J':>7} {'F':>7} {'ceiling':>8}   lost obj-frames")
for k in keys:
    s, (l, p) = SUM[k], lost(k, common)
    print(f"{k[0]:<20} {k[1]:<9} {str(GRIDS_E[k[0]]):>10} {s['J&F-Mean']:>7.4f} {s['J-Mean']:>7.4f} {s['F-Mean']:>7.4f} "
          f"{ceiling(GRIDS_E[k[0]], common):>8.3f}   {l:>5}/{p:<5} ({100 * l / max(p, 1):.1f}%)")
for k in keys:
    if len(clips_of[k]) > len(common):
        print(f"  ({label(k)} also finished {[c for c in clips_of[k] if c not in common]}: not in the table, for a fair comparison)")
for n, why in SKIPPED_E.items():
    print(f"SKIPPED {n}: {why}")
print(f"ceiling = best possible on that grid (perfect propagation); dino_ref predicts at 14× its grid, so its ceiling is approximate.\n"
      f"published: DINO ViT-B/16 J&F {E_REF_PAPER} on all 30 clips (different model and clips: order of magnitude only)")

# ── figure 1: official J&F per variant, ceiling ticks ──
fig, ax = plt.subplots(figsize=(max(8, 1.3 * len(keys)), 4.6))
x = np.arange(len(keys))
bars = ax.bar(x, [SUM[k]["J&F-Mean"] for k in keys], 0.7, color=[colors[k[1]] for k in keys])
ax.bar_label(bars, fmt="%.2f", fontsize=8)
for i, k in enumerate(keys):
    ax.plot([i - 0.35, i + 0.35], [ceiling(GRIDS_E[k[0]], common)] * 2, "k-", lw=2)
ax.axhline(E_REF_PAPER, color="tab:purple", ls="--", lw=1, label=f"DINO ViT-B/16, paper, 30 clips ({E_REF_PAPER})")
ax.plot([], [], "k-", lw=2, label="ceiling of that grid (perfect propagation)")
handles = ax.get_legend_handles_labels()[0] + [Patch(color=v, label=c if c != "none" else "DINO protocol (no upsampler)")
                                               for c, v in colors.items()]
ax.set_xticks(x); ax.set_xticklabels([f"{k[0]}\n{k[1]}\n{GRIDS_E[k[0]][0]}×{GRIDS_E[k[0]][1]}" for k in keys], fontsize=7)
ax.set_ylim(0, 1.05); ax.set_ylabel("official J&F"); ax.grid(axis="y", alpha=0.3)
ax.legend(handles=handles, fontsize=7, loc="upper left", bbox_to_anchor=(1.01, 1))
ax.set_title(f"Test E: DINOv2-B on {len(common)} clips — resolution ladder vs the DINO protocol")
plt.tight_layout(); plt.savefig(OUT_DIR / "E_bars.png", dpi=130, bbox_inches="tight"); plt.show()

# ── figure 2: per-clip J&F across the ladder (one line per clip and condition) ──
ups = [n for n, _, _ in E_VARIANTS if any(k[0] == n for k in keys)]
fig, axes = plt.subplots(1, len(E_CONDITIONS), figsize=(5.6 * len(E_CONDITIONS), 4.6), squeeze=False)
for ax, c in zip(axes[0], E_CONDITIONS):
    for clip in common:
        ys = [clip_mean_jf(RES_E[(n, c)][clip]) for n in ups]
        ax.plot(range(len(ups)), ys, "-o", ms=4, label=clip)
        if ("dino_ref", "none") in RES_E:
            ax.plot(len(ups), clip_mean_jf(RES_E[("dino_ref", "none")][clip]), "s", ms=5, color=ax.lines[-1].get_color())
    ax.set_xticks(range(len(ups) + (("dino_ref", "none") in RES_E)))
    ax.set_xticklabels([f"{n}\n{GRIDS_E[n][0]}×{GRIDS_E[n][1]}" for n in ups] + (["DINO\nprotocol"] if ("dino_ref", "none") in RES_E else []), fontsize=8)
    ax.set_ylim(0, 1); ax.grid(alpha=0.3); ax.set_ylabel("clip J&F"); ax.set_title(c, fontweight="bold")
    ax.legend(fontsize=7)
fig.suptitle("Test E: each line = one clip, from coarse to fine grid (square = DINO protocol)", fontsize=11)
plt.tight_layout(); plt.savefig(OUT_DIR / "E_per_clip.png", dpi=130, bbox_inches="tight"); plt.show()

# ── figure 3 (one per clip): GT and every variant at SHOW_FRACS of the clip ──
for clip in common:
    T = len(davis.frame_paths(ROOT, clip))
    ts = sorted({min(T - 1, int(round(f * (T - 1)))) for f in SHOW_FRACS})
    fig, axes = plt.subplots(len(ts), 1 + len(keys), figsize=(2.9 * (1 + len(keys)), 1.9 * len(ts) + 0.9), squeeze=False)
    for r, t in enumerate(ts):
        img, gt_s, hw = frame_and_gt(clip, t)
        panels = [(viz.overlay(img, gt_s), f"ground truth\nframe {t}/{T - 1}")]
        for k in keys:
            panels.append((viz.overlay(img, davis.resize_labels(PREDS_E[k][clip][t], hw)),
                           f"{label(k)}\nclip J&F {clip_mean_jf(RES_E[k][clip]):.3f}"))
        for ax, (p, cap) in zip(axes[r], panels):
            ax.imshow(p); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(cap, fontsize=7)
    fig.suptitle(f"Test E · {clip}", fontsize=11)
    plt.tight_layout(); plt.savefig(OUT_DIR / f"E_masks_{clip}.png", dpi=120, bbox_inches="tight"); plt.show()

# ── save ──
rows = [dict(variant=k[0], condition=k[1], clip=clip, grid=str(GRIDS_E[k[0]]), J=v["J"], F=v["F"], JF=v["JF"], n_objects=v["n_objects"])
        for k in keys for clip, v in metrics.summarize_jf({c: RES_E[k][c] for c in clips_of[k]})["per_clip"].items()]
summary = dict(common_clips=common,
               variants={f"{k[0]}/{k[1]}": dict(**{m: SUM[k][m] for m in ("J&F-Mean", "J-Mean", "F-Mean", "n_clips", "n_objects")},
                                                 ceiling=ceiling(GRIDS_E[k[0]], common), lost=list(lost(k, common))) for k in keys},
               grids={n: list(v) for n, v in GRIDS_E.items()}, skipped=SKIPPED_E)
config = dict(clips=CLIPS, E_CONDITIONS=E_CONDITIONS, E_VARIANTS=E_VARIANTS, TAU=TAU, TOP_K=TOP_K, N_LAST=N_LAST, E_CHUNK=E_CHUNK,
              E_Q_CHUNK=E_Q_CHUNK, E_REF=E_REF, SEED=SEED, gpu=GPU_NAME, repo_branch=REPO_BRANCH)
csv_p, json_p = metrics.save_results(OUT_DIR, rows, summary, config, name="exp3_debug_E")
print("saved:", csv_p, json_p, "+ figures in", OUT_DIR)